In [ ]:
import cl
import tables
import numpy as np
import pathlib
from common_cl_code import running_on_real_device


from common_cl_code import baseline_fr_file, recording_base_path

In [ ]:
overwrite = False
recording_length_minutes = 30

In [ ]:
frames_per_second = 25_000

In [ ]:
if not baseline_fr_file.exists() or overwrite:
    import datetime, pytz; print(datetime.datetime.now(pytz.timezone('US/Eastern')))
    with cl.open() as neurons:
        recording = neurons.record(
            stop_after_seconds=60*recording_length_minutes,
            include_spikes = True,
            include_stims = True,
            include_raw_samples=False,
            include_data_streams=False,
            file_location=recording_base_path
        )

        if running_on_real_device:
            recording.wait_until_stopped()
        else:
            for tick in neurons.loop(ticks_per_second=10, stop_after_seconds=60*recording_length_minutes+1):
                pass
            recording.stop()

        attrs = recording.attributes
        recording_path = recording.file['path']
    
    print(f"Recorded {attrs['duration_frames']} frames ({attrs['duration_seconds']} seconds)")
    print(f"into file: {recording.file['path']}")

    del recording
    with tables.open_file(recording_path) as h5_file:
        counts, bins = np.histogram(h5_file.root.spikes.col('channel'), bins = np.arange(65))
    
    baseline_fr = counts / (attrs['duration_frames'] / frames_per_second)

    assert attrs['frames_per_second'] == frames_per_second

    np.savez(baseline_fr_file, baseline_fr=baseline_fr, recording_path=recording_path, duration_frames=attrs['duration_frames'], allow_pickle=False)

In [ ]:
file_data = np.load(baseline_fr_file)
baseline_fr = file_data['baseline_fr']
recording_path = pathlib.Path(str(file_data['recording_path']))
duration_frames = file_data['duration_frames']

In [ ]:
import matplotlib.pyplot as plt
plt.plot(baseline_fr, '.')
plt.ylim(bottom=0)

In [ ]:
plt.matshow(baseline_fr.reshape(8,8).T)

In [ ]:
with tables.open_file(recording_path) as h5_file:
    t = h5_file.root.spikes.col('timestamp')
    c = h5_file.root.spikes.col('channel')
    print(h5_file)

In [ ]:
plt.scatter(t / frames_per_second, c, marker='|', color='k')